# CRIVO — Pointer/Copy Causal v5

**Pergunta:** o v4 mostrou que as cabeças aprendem as tarefas (no dev, AB chegou a 75–83% em cópia+fim e ABC a 50–83% nas seis tarefas), mas todas estragavam as respostas comuns do modelo, e por isso só o passo 1 ficava elegível. Ancorando a cabeça na própria base em perguntas comuns, ela aprende as tarefas **sem** mudar o resto?

**Única mudança causal em relação ao v4:** perda KL(base → cabeça) nas continuações que a própria base congelada gera para 800 mensagens de usuário do CRIVO (`curriculo_dialogo.json` e `curriculo_linguagem_neural.json`, split de treino; congeladas em `experimentos/pointer_copy_causal_v5_20261012/ancoras_v5.json`). A cabeça aprende a ficar desligada fora das tarefas.

- Arquiteturas do v4, sem alteração: **AB** (troca de espaço + parada) e **ABC** (AB + adaptador). A variante A sozinha saiu: o v4 mostrou que ela não consegue parar.
- Peso da âncora 1 e 4 → 4 configurações, mesmos lotes e hiperparâmetros.
- Preservação usada na escolha: 48 âncoras de dev. As 12 perguntas fixas do v2/v3 ficam só para a checagem final.
- **O candidato é escolhido só pelo dev; só ele abre o blind.** Mesmos dados congelados e gates do v3/v4.

Base `c0a56b15…`, fingerprint `257a4eb4…`, Transformer e `lm_head` congelados. Nada das rodadas anteriores é carregado ou sobrescrito; saída em `experimento_pointer_copy_v5/`.

Mesmo que todos os gates passem: `ativo_no_chat=false`, `promovido=false`, porta conversacional pendente.

> **Ambiente de execução → Executar tudo** em um Colab com GPU.


In [ ]:
# Célula 0 — ambiente, GPU, Drive, dependências
import os, sys, gc, json, math, time, hashlib, shutil, traceback, re, itertools, copy, random
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("PRECHECK 0/9 — ambiente", flush=True)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as exc:
    raise SystemExit(f"ERRO: PyTorch indisponível: {exc}")

try:
    from google.colab import drive
except Exception as exc:
    raise SystemExit(
        "ERRO: este notebook foi preparado para Google Colab. "
        f"google.colab não está disponível: {exc}"
    )

if not torch.cuda.is_available():
    raise SystemExit(
        "ERRO: GPU não disponível. No Colab: Ambiente de execução → Alterar tipo de ambiente → GPU."
    )

print(f"GPU OK: {torch.cuda.get_device_name(0)}", flush=True)
print(f"PyTorch={torch.__version__} CUDA={torch.version.cuda}", flush=True)
torch.backends.cuda.matmul.allow_tf32 = True

print("Montando Google Drive...", flush=True)
drive.mount("/content/drive", force_remount=False)
print("Drive montado.", flush=True)

try:
    import tokenizers
    from tokenizers import Tokenizer
except Exception:
    import subprocess
    print("Instalando tokenizers...", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1"])
    import tokenizers
    from tokenizers import Tokenizer

print(f"tokenizers={tokenizers.__version__}", flush=True)

SEED = 20261012
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("PRECHECK 0/9 OK", flush=True)


In [ ]:
# Célula 1 — clone da branch do experimento e verificação dos dados congelados
import subprocess, importlib

print("PRECHECK 1/9 — repositório e dados congelados", flush=True)
REPO_URL = "https://github.com/HROSONE/CRIVO"
REPO_BRANCH = "claude/affectionate-carson-22yoe0"
REPO_DIR = Path("/content/CRIVO")
EXP_REL = Path("experimentos/pointer_copy_causal_v3_20261012")

# SHA-256 dos arquivos congelados antes de qualquer treino (ver manifesto.json no repo).
ESPERADO_SHA = {
    "blind_novo_v3.json": "7a188ddfd948e38193dd55b1749513f63272b1ce04863cda7e6ed42022d984c8",
    "blind_regressao_v2.json": "c8aeec37b4beaa935cdb2c498717308b9dc155a6dbda3a1a0f6cb9801e937756",
    "treino_dev_v3.json": "4fb0edeb47742f7a3eb849e2bab76e22da59cf38c20bf752f3a039af8202edda",
    "dados_v3.py": "43aa9bb78fc183f90213d9ce023a241b46483c6ca44dbbceb8fd393ea0b1118e",
}

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
r = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
    capture_output=True, text=True,
)
if r.returncode != 0:
    raise SystemExit(f"ERRO ao clonar {REPO_URL}@{REPO_BRANCH}:\n{r.stdout}\n{r.stderr}")

commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
EXP_DIR = REPO_DIR / EXP_REL

def sha256_arquivo(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

for nome, esperado in ESPERADO_SHA.items():
    obtido = sha256_arquivo(EXP_DIR / nome)
    if obtido != esperado:
        raise SystemExit(f"ERRO: {nome} difere do congelado.\nesperado={esperado}\nobtido={obtido}")

EXP5_DIR = REPO_DIR / "experimentos/pointer_copy_causal_v5_20261012"
ANCORAS_SHA = "e7a9529dd1ad81e9e61722a95e9ebcd13361bea128011da08bda71c77c876bad"
if sha256_arquivo(EXP5_DIR / "ancoras_v5.json") != ANCORAS_SHA:
    raise SystemExit("ERRO: ancoras_v5.json difere do congelado")
ESPERADO_SHA["ancoras_v5.json"] = ANCORAS_SHA

for p in (str(REPO_DIR), str(EXP_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)
import linguagem_profunda as lp
import dados_v3 as d3
importlib.reload(d3)

esperados = ["<pad>", "<documento>", "<usuario>", "<assistente>", "<fim>"]
if list(lp.ESPECIAIS) != esperados:
    raise SystemExit(f"ERRO: especiais do repo mudaram: {lp.ESPECIAIS}")

print("commit:", commit, flush=True)
print("dados congelados conferidos por SHA-256:", list(ESPERADO_SHA), flush=True)
print("PRECHECK 1/9 OK", flush=True)


In [ ]:
# Célula 2 — mesma base do v2, conferida por SHA, copiada para o SSD
print("PRECHECK 2/9 — checkpoint", flush=True)

BASE_SHA_ESPERADO = "c0a56b1546495efafde5df99feebfa084fef414d512076d888b40c9828e4eb4e"
DRIVE_CHECKPOINTS = Path("/content/drive/MyDrive/CRIVO/checkpoints")
BASE_RUN = DRIVE_CHECKPOINTS / "escola_curricular_longa" / "run_transferido_20261012"
ORIGEM = BASE_RUN / "05_05_dialogo_estado"

for nome in ("pesos.pt", "tokenizer.json"):
    if not (ORIGEM / nome).is_file():
        raise SystemExit(f"ERRO: não encontrei {ORIGEM / nome}")

LOCAL = Path("/content/crivo_pointer_v5_base")
if LOCAL.exists():
    shutil.rmtree(LOCAL)
LOCAL.mkdir(parents=True)
for nome in ("pesos.pt", "tokenizer.json"):
    print(f"Copiando {nome} para SSD...", flush=True)
    shutil.copy2(ORIGEM / nome, LOCAL / nome)

SHA_PESOS = sha256_arquivo(LOCAL / "pesos.pt")
SHA_TOKENIZER = sha256_arquivo(LOCAL / "tokenizer.json")
if SHA_PESOS != BASE_SHA_ESPERADO:
    raise SystemExit(f"ERRO: a base não é a mesma do v2.\nesperado={BASE_SHA_ESPERADO}\nobtido={SHA_PESOS}")
print("Base:", BASE_RUN, flush=True)
print("pesos.pt sha256:", SHA_PESOS, "(igual ao v2, v3 e v4)", flush=True)
print("PRECHECK 2/9 OK", flush=True)


In [ ]:
# Célula 3 — carregar, congelar e fingerprint da base
print("PRECHECK 3/9 — modelo/tokenizer", flush=True)

try:
    modelo, tokenizer, estado = lp.carregar(LOCAL, dispositivo="cpu")
except Exception as exc:
    raise SystemExit(f"ERRO ao carregar checkpoint: {exc}\n{traceback.format_exc()}")

cfg = modelo.config
if cfg.vocabulario != 8192 or cfg.dimensao != 384 or cfg.camadas != 8 or cfg.cabecas != 8:
    print(
        "AVISO: configuração difere do snapshot esperado "
        f"(vocab={cfg.vocabulario}, dim={cfg.dimensao}, camadas={cfg.camadas}, cabecas={cfg.cabecas}).",
        flush=True,
    )

def fingerprint_modulo(mod: nn.Module) -> str:
    h = hashlib.sha256()
    with torch.no_grad():
        for nome, param in sorted(mod.named_parameters(), key=lambda x: x[0]):
            t = param.detach().cpu().contiguous()
            h.update(nome.encode("utf-8"))
            h.update(str(t.dtype).encode("ascii"))
            h.update(str(tuple(t.shape)).encode("ascii"))
            h.update(t.numpy().tobytes())
    return h.hexdigest()

modelo.eval()
for p in modelo.parameters():
    p.requires_grad_(False)
    p.grad = None

N_BASE = sum(p.numel() for p in modelo.parameters())
HASH_ANTES = fingerprint_modulo(modelo)
FINGERPRINT_V2 = "257a4eb41406b9aeeee8f9cf0471c1d8423f769685643200d03c6886a1ea1566"
if HASH_ANTES != FINGERPRINT_V2:
    raise SystemExit(f"ERRO: fingerprint da base difere do v2: {HASH_ANTES}")

ids_especiais = {}
for s in lp.ESPECIAIS:
    tid = tokenizer.token_to_id(s)
    if tid is None:  # noqa
        raise SystemExit(f"ERRO: especial ausente no tokenizer: {s}")
    ids_especiais[s] = int(tid)

ID_PAD = ids_especiais["<pad>"]
ID_USUARIO = ids_especiais["<usuario>"]
ID_ASSISTENTE = ids_especiais["<assistente>"]
ID_FIM = ids_especiais["<fim>"]
ESPECIAIS_SET = set(ids_especiais.values())
PROIBIDOS_GERACAO = [i for i in ESPECIAIS_SET if i != ID_FIM]

dispositivo = torch.device("cuda")
modelo = modelo.to(dispositivo).eval()

print(
    f"Config: vocab={cfg.vocabulario} dim={cfg.dimensao} camadas={cfg.camadas} "
    f"cabecas={cfg.cabecas} contexto={cfg.contexto} dropout={cfg.dropout}",
    flush=True
)
print(f"Base: {N_BASE:,} parâmetros — todos congelados", flush=True)
print("IDs especiais:", ids_especiais, flush=True)
print("base_hash_before:", HASH_ANTES, flush=True)
print("pesos.pt sha256:", SHA_PESOS, flush=True)
print("tokenizer sha256:", SHA_TOKENIZER, flush=True)
print("PRECHECK 3/9 OK", flush=True)


In [ ]:
# Célula 4 — cabeça v4 (inalterada) + configurações do v5
print("PRECHECK 4/9 — cabeça v4", flush=True)

@torch.no_grad()
def estados_congelados(ids: torch.Tensor) -> torch.Tensor:
    if ids.ndim != 2:
        raise ValueError("ids deve ser [B,T]")
    if ids.shape[1] > cfg.contexto:
        raise ValueError(f"sequência {ids.shape[1]} > contexto {cfg.contexto}")
    pos = torch.arange(ids.shape[1], device=ids.device)
    x = modelo.embedding(ids) + modelo.posicao(pos)
    for bloco in modelo.blocos:
        x = bloco(x)
    return modelo.norm(x)

def mascara_fonte(ids: torch.Tensor, prompt_lens: torch.Tensor) -> torch.Tensor:
    """True só nas posições do prompt ORIGINAL que não são especiais (igual ao v2)."""
    b, t = ids.shape
    pos = torch.arange(t, device=ids.device).unsqueeze(0)
    src = pos < prompt_lens.unsqueeze(1)
    for sid in ESPECIAIS_SET:
        src &= ids.ne(sid)
    return src

def construir_mapa_variante():
    """Para cada ID, o ID da mesma palavra com/sem o 'Ġ' inicial, tirado do próprio vocabulário."""
    mapa = list(range(cfg.vocabulario))
    trocas = 0
    for tid in range(cfg.vocabulario):
        if tid in ESPECIAIS_SET:
            continue
        tok = tokenizer.id_to_token(tid)
        outro = tokenizer.token_to_id(tok[1:]) if tok.startswith("Ġ") and len(tok) > 1 else tokenizer.token_to_id("Ġ" + tok)
        if outro is not None and outro not in ESPECIAIS_SET:
            mapa[tid] = int(outro)
            trocas += 1
    return torch.tensor(mapa, dtype=torch.long), trocas

MAPA_VARIANTE_CPU, N_TROCAS = construir_mapa_variante()
MAPA_VARIANTE = MAPA_VARIANTE_CPU.to(dispositivo)

class PointerCopyV4(nn.Module):
    """q/k/gate idênticos ao v2; rotas novas começam neutras (não mudam a saída no passo 0)."""

    def __init__(self, dimensao, vocabulario, rank=32, p_gen_init=0.90,
                 variante=False, parada=False, adaptador_rank=0):
        super().__init__()
        self.rank = int(rank)
        self.vocabulario = int(vocabulario)
        self.usa_variante, self.usa_parada = bool(variante), bool(parada)
        self.adaptador_rank = int(adaptador_rank)
        self.q = nn.Linear(dimensao, rank, bias=False)
        self.k = nn.Linear(dimensao, rank, bias=False)
        self.gate = nn.Linear(dimensao, 1)
        nn.init.normal_(self.q.weight, std=0.02)
        nn.init.normal_(self.k.weight, std=0.02)
        nn.init.zeros_(self.gate.weight)
        nn.init.constant_(self.gate.bias, math.log(p_gen_init / (1.0 - p_gen_init)))
        if self.usa_variante:  # (A) quanto da massa de cópia vai para a variante com/sem 'Ġ'
            self.troca = nn.Linear(dimensao, 1)
            nn.init.zeros_(self.troca.weight)
            nn.init.zeros_(self.troca.bias)
        if self.usa_parada:  # (B) probabilidade explícita de emitir <fim>
            self.parada = nn.Linear(dimensao, 1)
            nn.init.zeros_(self.parada.weight)
            nn.init.constant_(self.parada.bias, math.log(0.01 / 0.99))
        if self.adaptador_rank:  # (C) correção low-rank dos logits; começa em zero
            self.ad_down = nn.Linear(dimensao, self.adaptador_rank, bias=False)
            self.ad_up = nn.Linear(self.adaptador_rank, vocabulario, bias=False)
            nn.init.normal_(self.ad_down.weight, std=0.02)
            nn.init.zeros_(self.ad_up.weight)

    def _espalhar(self, attn, ids):
        b, t = ids.shape
        p = torch.zeros((b, t, self.vocabulario), dtype=attn.dtype, device=attn.device)
        p.scatter_add_(2, ids.unsqueeze(1).expand(b, t, t), attn)
        return p

    def forward(self, h, ids, source_mask):
        b, t, _ = h.shape
        q, k = self.q(h), self.k(h)
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.rank)
        causal = torch.tril(torch.ones((t, t), dtype=torch.bool, device=h.device))
        valido = causal.unsqueeze(0) & source_mask.unsqueeze(1)
        scores = scores.masked_fill(~valido, -1e4)
        attn = torch.softmax(scores, dim=-1)
        tem_fonte = valido.any(dim=-1, keepdim=True)
        attn = torch.where(tem_fonte, attn, torch.zeros_like(attn))
        attn = attn * valido.to(attn.dtype)
        attn = attn / attn.sum(dim=-1, keepdim=True).clamp_min(1e-8)

        p_copy = self._espalhar(attn, ids)
        troca = None
        if self.usa_variante:
            troca = torch.sigmoid(self.troca(h))
            p_copy = (1.0 - troca) * p_copy + troca * self._espalhar(attn, MAPA_VARIANTE[ids])
        p_copy = p_copy.clone()
        p_copy[:, :, list(ESPECIAIS_SET)] = 0.0
        soma = p_copy.sum(dim=-1, keepdim=True)
        p_copy = torch.where(soma > 0, p_copy / soma.clamp_min(1e-8), torch.zeros_like(p_copy))

        p_gen = torch.sigmoid(self.gate(h))
        p_stop = torch.sigmoid(self.parada(h)) if self.usa_parada else None
        delta = self.ad_up(self.ad_down(h)) if self.adaptador_rank else None
        return {"p_copy": p_copy, "p_gen": p_gen, "attn": attn, "tem_fonte": tem_fonte,
                "p_stop": p_stop, "delta": delta, "troca": troca}

def distribuicao_final(pointer_obj, ids, prompt_lens):
    modelo.eval()
    with torch.no_grad():
        h = estados_congelados(ids)
        logits_base = F.linear(h, modelo.embedding.weight)
    h = h.detach()
    src = mascara_fonte(ids, prompt_lens)
    out = pointer_obj(h, ids, src)
    logits = logits_base if out["delta"] is None else logits_base + out["delta"]
    if PROIBIDOS_GERACAO:
        logits = logits.clone()
        logits[:, :, PROIBIDOS_GERACAO] = -1e9
        logits_base = logits_base.clone()
        logits_base[:, :, PROIBIDOS_GERACAO] = -1e9
    p_vocab = torch.softmax(logits, dim=-1)
    out["p_base"] = torch.softmax(logits_base, dim=-1)
    p_gen_eff = torch.where(out["tem_fonte"], out["p_gen"], torch.ones_like(out["p_gen"]))
    p_final = p_gen_eff * p_vocab + (1.0 - p_gen_eff) * out["p_copy"]
    if out["p_stop"] is not None:
        fim = torch.zeros_like(p_final)
        fim[:, :, ID_FIM] = 1.0
        p_final = (1.0 - out["p_stop"]) * p_final + out["p_stop"] * fim
    p_final = p_final / p_final.sum(dim=-1, keepdim=True).clamp_min(1e-8)
    return p_final, p_vocab, out, src

RANK = 32
ADAPTADOR_RANK = 32
VARIANTES = {
    "AB_troca_parada": dict(variante=True, parada=True, adaptador_rank=0),
    "ABC_troca_parada_adaptador": dict(variante=True, parada=True, adaptador_rank=ADAPTADOR_RANK),
}
# v5: única mudança causal = âncora KL na própria base; dois pesos, escolhidos pelo dev.
CONFIGS = {
    "AB_kl1": ("AB_troca_parada", 1.0),
    "AB_kl4": ("AB_troca_parada", 4.0),
    "ABC_kl1": ("ABC_troca_parada_adaptador", 1.0),
    "ABC_kl4": ("ABC_troca_parada_adaptador", 4.0),
}

def nova_cabeca(nome):
    arq = CONFIGS[nome][0] if nome in CONFIGS else nome
    return PointerCopyV4(cfg.dimensao, cfg.vocabulario, rank=RANK, **VARIANTES[arq]).to(dispositivo)

def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

# Teste estrutural: no passo 0, (B) e (C) quase não mexem na distribuição.
_ids = torch.randint(5, cfg.vocabulario, (2, 9), device=dispositivo)
_ids[:, 0] = ID_USUARIO
_pl = torch.tensor([6, 5], device=dispositivo)
for _nome in VARIANTES:
    _m = nova_cabeca(_nome)
    with torch.no_grad():
        _pf, _pv, _o, _s = distribuicao_final(_m, _ids, _pl)
    assert _pf.shape == (2, 9, cfg.vocabulario) and torch.isfinite(_pf).all()
    assert float(_o["p_copy"][:, :, list(ESPECIAIS_SET)].abs().max()) == 0.0
    assert torch.allclose(_pf.sum(-1), torch.ones_like(_pf.sum(-1)), atol=1e-4)
    print(f"  {_nome}: {n_params(_m):,} parâmetros treináveis ({100*n_params(_m)/N_BASE:.3f}% da base)", flush=True)
del _ids, _pl, _m, _pf, _pv, _o, _s

print(f"Mapa com/sem 'Ġ': {N_TROCAS:,} de {cfg.vocabulario:,} IDs têm variante no vocabulário", flush=True)
print("PRECHECK 4/9 OK", flush=True)


In [ ]:
# Célula 5 — dados congelados; re-tokenização no runtime tem de bater byte a byte
print("PRECHECK 5/9 — dados", flush=True)

def ler_json(nome):
    return json.loads((EXP_DIR / nome).read_text(encoding="utf-8"))

BLIND_NOVO = ler_json("blind_novo_v3.json")
BLIND_V2 = ler_json("blind_regressao_v2.json")
TREINO_DEV = ler_json("treino_dev_v3.json")

def remontar(entidades):
    casos = []
    for i, ent in enumerate(entidades):
        for j, tarefa in enumerate(d3.TAREFAS):
            casos.append(d3.montar_caso(lp, tokenizer, ent, tarefa, d3.POSICOES[(i + j) % 3], cfg.contexto))
    return casos

def conferir_igual(montados, congelados, rotulo):
    if len(montados) != len(congelados):
        raise SystemExit(f"ERRO: {rotulo} tem {len(montados)} casos; congelado tem {len(congelados)}")
    for a, b in zip(montados, congelados):
        if a["prompt_ids"] != b["prompt_ids"] or a["resp_ids"] != b["resp_ids"]:
            raise SystemExit(f"ERRO: tokenização do runtime divergiu do congelado em {rotulo}: {a['nome']}/{a['tarefa']}")

BLIND_NOVO_EX = remontar(BLIND_NOVO["entidades"])
conferir_igual(BLIND_NOVO_EX, BLIND_NOVO["casos"], "blind novo")
BLIND_V2_REAL_EX = remontar(BLIND_V2["entidades"])
conferir_igual(BLIND_V2_REAL_EX, BLIND_V2["casos_formato_real"], "blind v2 formato real")
BLIND_V2_LEGADO_EX = BLIND_V2["casos_legado_v2"]  # formato antigo (IDs colados) só para regressão

rr = random.Random(SEED + 2002)
TREINO = []
for i, ent in enumerate(TREINO_DEV["treino"]):
    for j, tarefa in enumerate(d3.TAREFAS):
        pos = d3.POSICOES[(i + j + rr.randrange(3)) % 3]
        TREINO.append(d3.montar_caso(lp, tokenizer, ent, tarefa, pos, cfg.contexto))
rr.shuffle(TREINO)

DEV_EX = []  # v4: dev com as 6 tarefas, para escolher a variante sem olhar o blind
for i, ent in enumerate(TREINO_DEV["dev"]):
    for j, tarefa in enumerate(d3.TAREFAS):
        DEV_EX.append(d3.montar_caso(lp, tokenizer, ent, tarefa, d3.POSICOES[(i + j) % 3], cfg.contexto))

NO_COPY_EX = [d3.montar_no_copy(lp, tokenizer, q, a, cfg.contexto) for q, a in d3.NO_COPY_PARES]

def nomes(xs):
    return {x["nome"].casefold() for x in xs}

conjuntos = {
    "treino": TREINO_DEV["treino"], "dev": TREINO_DEV["dev"],
    "blind_v2": BLIND_V2["entidades"], "blind_novo": BLIND_NOVO["entidades"],
}
for a in conjuntos:
    for b in conjuntos:
        if a < b and nomes(conjuntos[a]) & nomes(conjuntos[b]):
            raise SystemExit(f"ERRO: interseção entre {a} e {b}")

def resumo_split(xs):
    return {"n": len(xs), "grupo": dict(Counter(x["grupo"] for x in xs)),
            "tokens": dict(sorted(Counter(x["n_tokens"] for x in xs).items()))}

def copiabilidade(casos):
    out = {}
    for t in d3.TAREFAS:
        xs = [c for c in casos if c["tarefa"] == t]
        out[t] = {
            "primeiro_token_copiavel": sum(c["copy_resp"][c["nome_resp"].index(True)] for c in xs) / len(xs),
            "nome_inteiro_copiavel": sum(c["nome_todo_copiavel"] for c in xs) / len(xs),
        }
    return out

MAPA_LISTA = MAPA_VARIANTE_CPU.tolist()

def marcar_copia_v4(ex):
    """Token do nome é copiável no v4 se ele ou sua variante com/sem 'Ġ' está no prompt."""
    fonte = {t for t in ex["prompt_ids"] if t not in ESPECIAIS_SET}
    fonte |= {MAPA_LISTA[t] for t in fonte}
    ex["copy_resp_v3"] = list(ex["copy_resp"])
    ex["copy_resp"] = [m and t in fonte for t, m in zip(ex["resp_ids"], ex["nome_resp"])]
    ex["nome_todo_copiavel"] = all(c for c, m in zip(ex["copy_resp"], ex["nome_resp"]) if m)
    return ex

for _ex in TREINO + DEV_EX + BLIND_NOVO_EX + BLIND_V2_REAL_EX:
    marcar_copia_v4(_ex)

COPIABILIDADE = copiabilidade(BLIND_NOVO_EX)
print("treino:", len(TREINO), "dev:", len(DEV_EX), "no-copy:", len(NO_COPY_EX), flush=True)
print("blind novo:", len(BLIND_NOVO_EX), "| blind v2 real:", len(BLIND_V2_REAL_EX),
      "| blind v2 legado:", len(BLIND_V2_LEGADO_EX), flush=True)
for k, v in conjuntos.items():
    print(f"  {k}:", resumo_split(v), flush=True)
print("Copiabilidade v4 (ID ou variante com/sem Ġ) no blind novo:", json.dumps(COPIABILIDADE, ensure_ascii=False), flush=True)
print("Interseções entre treino/dev/blind_v2/blind_novo = 0", flush=True)
print("PRECHECK 5/9 OK", flush=True)


In [ ]:
# Célula 6 — batch, perda, geração e avaliação do v4
print("PRECHECK 6/9 — perda/geração/avaliação", flush=True)

def colar_batch(exemplos, device):
    fulls, labels_rows, copy_rows, prompt_lens, fim_rows = [], [], [], [], []
    for ex in exemplos:
        prompt = list(ex["prompt_ids"])
        resp = list(ex["resp_ids"])
        copy_resp = list(ex["copy_resp"])
        if len(resp) != len(copy_resp):
            raise ValueError("resp_ids/copy_resp com tamanhos diferentes")
        full = prompt + resp
        if len(full) > cfg.contexto:
            raise ValueError("batch contém sequência acima do contexto")

        labels = [-100] * len(prompt) + resp
        copy_target = [False] * len(prompt) + copy_resp
        fim_target = [False] * len(prompt) + [
            ex["tarefa"] == "copia" and i == len(resp) - 1 and tok == ID_FIM
            for i, tok in enumerate(resp)
        ]

        fulls.append(full)
        labels_rows.append(labels)
        copy_rows.append(copy_target)
        fim_rows.append(fim_target)
        prompt_lens.append(len(prompt))

    tmax = max(map(len, fulls))
    b = len(fulls)
    ids = torch.full((b, tmax), ID_PAD, dtype=torch.long, device=device)
    labels = torch.full((b, tmax), -100, dtype=torch.long, device=device)
    copy_targets = torch.zeros((b, tmax), dtype=torch.bool, device=device)
    fim_targets = torch.zeros((b, tmax), dtype=torch.bool, device=device)

    for i, (full, lab, cpy, fim) in enumerate(zip(fulls, labels_rows, copy_rows, fim_rows)):
        n = len(full)
        ids[i, :n] = torch.tensor(full, dtype=torch.long, device=device)
        labels[i, :n] = torch.tensor(lab, dtype=torch.long, device=device)
        copy_targets[i, :n] = torch.tensor(cpy, dtype=torch.bool, device=device)
        fim_targets[i, :n] = torch.tensor(fim, dtype=torch.bool, device=device)

    prompt_lens_t = torch.tensor(prompt_lens, dtype=torch.long, device=device)
    return ids, labels, copy_targets, prompt_lens_t, fim_targets

def perda_batch(pointer_obj, exemplos, device):
    ids, labels, copy_targets, prompt_lens, fim_targets = colar_batch(exemplos, device)
    p_final, _, out, src = distribuicao_final(pointer_obj, ids, prompt_lens)

    pred = p_final[:, :-1, :]
    copy_pred = out["p_copy"][:, :-1, :]
    gate_pred = out["p_gen"][:, :-1, 0]
    gold = labels[:, 1:]
    supervised = gold.ne(-100)
    if not supervised.any():
        raise RuntimeError("batch sem tokens supervisionados")
    copy_gold_mask = copy_targets[:, 1:] & supervised
    safe_gold = gold.clamp_min(0)

    mix_nll = -torch.log(pred.gather(-1, safe_gold.unsqueeze(-1)).squeeze(-1).clamp_min(1e-8))
    loss_mix = mix_nll[supervised].mean()

    if copy_gold_mask.any():
        copy_nll = -torch.log(copy_pred.gather(-1, safe_gold.unsqueeze(-1)).squeeze(-1).clamp_min(1e-8))
        loss_copy = copy_nll[copy_gold_mask].mean()
    else:
        loss_copy = torch.zeros((), device=device)

    gate_target = (~copy_gold_mask).float()
    loss_gate = F.binary_cross_entropy(gate_pred.clamp(1e-6, 1 - 1e-6), gate_target, reduction="none")[supervised].mean()

    fim_gold_mask = fim_targets[:, 1:] & supervised  # <fim> da cópia pura, explícito (igual ao v3)
    loss_fim = mix_nll[fim_gold_mask].mean() if fim_gold_mask.any() else torch.zeros((), device=device)

    loss = loss_mix + 0.50 * loss_copy + 0.10 * loss_gate + 0.50 * loss_fim
    loss_stop = torch.zeros((), device=device)
    if out["p_stop"] is not None:  # (B) parada supervisionada em todo <fim> de resposta
        stop_pred = out["p_stop"][:, :-1, 0]
        stop_target = gold.eq(ID_FIM).float()
        loss_stop = F.binary_cross_entropy(stop_pred.clamp(1e-6, 1 - 1e-6), stop_target, reduction="none")[supervised].mean()
        loss = loss + 0.10 * loss_stop

    with torch.no_grad():
        gen_mask = supervised & ~copy_gold_mask
        metricas = {
            "loss_mix": float(loss_mix.item()), "loss_copy": float(loss_copy.item()),
            "loss_gate": float(loss_gate.item()), "loss_fim": float(loss_fim.item()),
            "loss_stop": float(loss_stop.item()),
            "p_gen_copy": float(gate_pred[copy_gold_mask].mean().item()) if copy_gold_mask.any() else None,
            "p_gen_generate": float(gate_pred[gen_mask].mean().item()) if gen_mask.any() else None,
        }
    return loss, metricas

def perda_ancora(pointer_obj, exemplos, device):
    """KL(base || cabeça) nas continuações que a própria base gerou para as âncoras."""
    ids, labels, _, prompt_lens, _ = colar_batch(exemplos, device)
    p_final, _, out, _ = distribuicao_final(pointer_obj, ids, prompt_lens)
    pb, pf = out["p_base"][:, :-1], p_final[:, :-1]
    mask = labels[:, 1:].ne(-100)
    kl = (pb * (torch.log(pb.clamp_min(1e-9)) - torch.log(pf.clamp_min(1e-9)))).sum(-1)
    return kl[mask].mean()

@torch.no_grad()
def probs_base(ids):
    h = estados_congelados(ids)
    logits = F.linear(h, modelo.embedding.weight)
    p = torch.softmax(logits, dim=-1)
    if PROIBIDOS_GERACAO:
        p[:, :, PROIBIDOS_GERACAO] = 0.0
        p = p / p.sum(dim=-1, keepdim=True).clamp_min(1e-8)
    return p

@torch.no_grad()
def gerar_base(prompt_ids, max_novos=28):
    ids = list(prompt_ids)
    gerados = []
    limite = min(max_novos, max(0, cfg.contexto - len(ids)))
    for _ in range(limite):
        x = torch.tensor([ids], dtype=torch.long, device=dispositivo)
        p = probs_base(x)[0, -1]
        nxt = int(torch.argmax(p).item())
        if nxt == ID_FIM:
            return gerados, True
        if nxt in PROIBIDOS_GERACAO:
            raise RuntimeError("base tentou emitir token especial proibido após mascaramento")
        gerados.append(nxt)
        ids.append(nxt)
    return gerados, False

@torch.no_grad()
def gerar_pointer(pointer_obj, prompt_ids, max_novos=28):
    ids = list(prompt_ids)
    prompt_len = len(prompt_ids)
    gerados, pgens = [], []
    limite = min(max_novos, max(0, cfg.contexto - prompt_len))
    for _ in range(limite):
        x = torch.tensor([ids], dtype=torch.long, device=dispositivo)
        pl = torch.tensor([prompt_len], dtype=torch.long, device=dispositivo)
        p_final, _, out, src = distribuicao_final(pointer_obj, x, pl)
        if len(ids) > prompt_len and bool(src[0, prompt_len:].any().item()):
            raise RuntimeError("BUG: token gerado entrou na source mask do pointer")
        pgens.append(float(out["p_gen"][0, -1, 0].item()))
        nxt = int(torch.argmax(p_final[0, -1]).item())
        if nxt == ID_FIM:
            return gerados, True, sum(pgens) / len(pgens)
        if nxt in PROIBIDOS_GERACAO:
            raise RuntimeError("pointer tentou emitir especial proibido")
        gerados.append(nxt)
        ids.append(nxt)
    return gerados, False, (sum(pgens) / len(pgens) if pgens else None)

def max_novos_ex(ex):
    return {
        "copia": 8, "categoria": 18, "funcao": 22, "invariancia": 24,
        "composicao": 28, "negacao": 28, "no_copy": 12,
    }.get(ex["tarefa"], 24)

def avaliar_conjunto(modo, casos, pointer_obj=None, rotulo="eval"):
    """Formato real: métricas no texto decodificado (dados_v3.avaliar_saida)."""
    if modo not in ("base", "pointer"):
        raise ValueError(modo)
    print(f"Avaliando {rotulo}: {len(casos)} casos", flush=True)
    linhas, t0 = [], time.time()
    for i, ex in enumerate(casos, 1):
        if modo == "base":
            gerados, terminou = gerar_base(ex["prompt_ids"], max_novos=max_novos_ex(ex))
            pg = None
        else:
            gerados, terminou, pg = gerar_pointer(pointer_obj, ex["prompt_ids"], max_novos=max_novos_ex(ex))
        texto = tokenizer.decode(gerados)
        linha = {k: ex[k] for k in ("nome", "grupo", "n_tokens", "tarefa", "posicao",
                                    "nome_no_inicio_da_resposta", "nome_todo_copiavel")}
        linha.update(d3.avaliar_saida(ex, texto, terminou))
        linha.update({"p_gen_medio": pg, "alvo": ex["resposta"], "saida": texto.strip()})
        linhas.append(linha)
        if i == 1 or i % 100 == 0 or i == len(casos):
            print(f"  {rotulo}: {i}/{len(casos)} ({time.time()-t0:.1f}s)", flush=True)
    return linhas

def contem_subspan(hay, needle):
    n = len(needle)
    return bool(needle) and any(hay[i:i+n] == needle for i in range(len(hay) - n + 1))

def avaliar_legado(modo, casos, pointer_obj=None, rotulo="legado"):
    """Formato antigo do v2 (IDs colados) com as métricas do v2, mais nome+<fim>."""
    print(f"Avaliando {rotulo}: {len(casos)} casos", flush=True)
    linhas, t0 = [], time.time()
    for i, ex in enumerate(casos, 1):
        if modo == "base":
            gerados, terminou = gerar_base(ex["prompt_ids"], max_novos=max_novos_ex(ex))
        else:
            gerados, terminou, _ = gerar_pointer(pointer_obj, ex["prompt_ids"], max_novos=max_novos_ex(ex))
        nome_ok = contem_subspan(gerados, ex["nome_ids"])
        cat_ok = contem_subspan(gerados, ex["cat_ids"])
        fun_ok = contem_subspan(gerados, ex["fun_ids"])
        txt = tokenizer.decode(gerados).casefold()
        neg = ("não" in txt) or ("nao" in txt)
        sem = {"copia": gerados == ex["nome_ids"], "categoria": nome_ok and cat_ok,
               "funcao": nome_ok and fun_ok, "invariancia": nome_ok and neg,
               "composicao": nome_ok and cat_ok and fun_ok, "negacao": nome_ok and cat_ok and neg}[ex["tarefa"]]
        linhas.append({"tarefa": ex["tarefa"], "grupo": ex["grupo"], "entity_exact": nome_ok,
                       "pure_copy_exact": ex["tarefa"] == "copia" and gerados == ex["nome_ids"],
                       "pure_copy_exact_fim": ex["tarefa"] == "copia" and gerados == ex["nome_ids"] and terminou,
                       "semantic_ok": bool(sem), "terminou": bool(terminou),
                       "saida": tokenizer.decode(gerados)})
        if i == 1 or i % 100 == 0 or i == len(casos):
            print(f"  {rotulo}: {i}/{len(casos)} ({time.time()-t0:.1f}s)", flush=True)
    return linhas

def resumir_legado(linhas):
    def taxa(xs, k):
        return None if not xs else sum(bool(x[k]) for x in xs) / len(xs)
    copia = [r for r in linhas if r["tarefa"] == "copia"]
    out = {"n": len(linhas), "exact_entity_match": taxa(linhas, "entity_exact"),
           "pure_copy_exact_match": taxa(copia, "pure_copy_exact"),
           "pure_copy_exact_fim": taxa(copia, "pure_copy_exact_fim"),
           "lexical_close": taxa([r for r in linhas if r["grupo"] == "close"], "entity_exact"),
           "lexical_distant": taxa([r for r in linhas if r["grupo"] == "distant"], "entity_exact"),
           "termination_rate": taxa(linhas, "terminou")}
    for t in ("categoria", "funcao", "invariancia", "composicao", "negacao"):
        out[t] = taxa([r for r in linhas if r["tarefa"] == t], "semantic_ok")
    out["semantica_geral"] = sum(out[t] for t in ("categoria", "funcao", "invariancia", "composicao", "negacao")) / 5
    return out

@torch.no_grad()
def teto_teacher_forcing(casos, variante=False, parada=False):
    """Limite superior da mistura sem adaptador, sob teacher forcing.

    Um alvo é alcançável se a cópia pode emiti-lo (ID do prompt; com variante=True,
    também a variante com/sem 'Ġ'), se a parada pode emiti-lo (<fim>, com parada=True),
    ou se já é o top-1 da base congelada. Com o adaptador (C) tudo passa a ser alcançável.
    """
    linhas = []
    for ex in casos:
        prompt, resp = ex["prompt_ids"], ex["resp_ids"]
        x = torch.tensor([prompt + resp], dtype=torch.long, device=dispositivo)
        p = probs_base(x)[0]
        P = len(prompt)
        top1 = p[P - 1:P - 1 + len(resp)].argmax(-1).tolist()
        fonte = {t for t in prompt if t not in ESPECIAIS_SET}
        if variante:
            fonte |= {MAPA_LISTA[t] for t in fonte}
        alc = [(t in fonte) or (top1[i] == t) or (parada and t == ID_FIM) for i, t in enumerate(resp)]
        nome_pos = [i for i, m in enumerate(ex["nome_resp"]) if m]
        linhas.append({"tarefa": ex["tarefa"], "nome_alcancavel": all(alc[i] for i in nome_pos),
                       "resposta_alcancavel": all(alc)})
    out = {}
    for t in d3.TAREFAS:
        xs = [r for r in linhas if r["tarefa"] == t]
        out[t] = {k: sum(r[k] for r in xs) / len(xs) for k in ("nome_alcancavel", "resposta_alcancavel")}
    out["teto_exact_name_fim_copia"] = out["copia"]["resposta_alcancavel"]
    return out

print("PRECHECK 6/9 OK", flush=True)


In [ ]:
# Célula 7 — SANITY: backward + passos em cada variante, base sem gradiente, geração real
print("PRECHECK 7/9 — âncoras e SANITY/SMOKE", flush=True)

ANCORAS = json.loads((EXP5_DIR / "ancoras_v5.json").read_text(encoding="utf-8"))

def caso_ancora(texto):
    prompt = list(lp.fonte_dialogo(tokenizer, texto, (), cfg.contexto))
    gerados, terminou = gerar_base(prompt, max_novos=24)
    resp = gerados + ([ID_FIM] if terminou else [])
    if not resp:
        resp = [ID_FIM]
    return {"tarefa": "ancora", "mensagem": texto, "prompt_ids": prompt, "resp_ids": resp,
            "copy_resp": [False] * len(resp), "nome_resp": [False] * len(resp), "no_copy": True}

t0 = time.time()
ANCORA_TREINO = [caso_ancora(t) for t in ANCORAS["treino"]]
ANCORA_DEV_PROMPTS = list(ANCORAS["dev"])
print(f"Âncoras: {len(ANCORA_TREINO)} continuações da base geradas em {time.time()-t0:.1f}s; "
      f"{len(ANCORA_DEV_PROMPTS)} prompts de dev para preservação", flush=True)

assert len(TREINO_DEV["treino"]) == 480 and len(TREINO_DEV["dev"]) == 80
assert len(BLIND_NOVO_EX) == 600 and len(BLIND_V2_REAL_EX) == 600 and len(BLIND_V2_LEGADO_EX) == 600
assert len(DEV_EX) == 480
assert all(ex["prompt_ids"][-2:] == [ID_FIM, ID_ASSISTENTE] for ex in BLIND_NOVO_EX + TREINO[:200])
assert all(ex["resp_ids"][-1] == ID_FIM for ex in BLIND_NOVO_EX + TREINO)
_x = colar_batch([x for x in TREINO if x["tarefa"] == "copia"][:4], dispositivo)
assert int(_x[4].sum().item()) == 4, "máscara do <fim> da cópia pura não marcou 1 posição por exemplo"
del _x

rr_s = random.Random(SEED + 3003)
for nome_var in VARIANTES:
    cab = nova_cabeca(nome_var)
    opt = torch.optim.AdamW(cab.parameters(), lr=1e-3, weight_decay=0.0)
    for passo in range(1, 4):
        batch = rr_s.sample(TREINO, 8) + [rr_s.choice(NO_COPY_EX) for _ in range(4)]
        opt.zero_grad(set_to_none=True)
        loss_s, met_s = perda_batch(cab, batch, dispositivo)
        kl_s = perda_ancora(cab, rr_s.sample(ANCORA_TREINO, 4), dispositivo)
        if not torch.isfinite(kl_s):
            raise SystemExit(f"ERRO SANITY {nome_var}: KL de âncora não finita")
        loss_s = loss_s + kl_s
        if not torch.isfinite(loss_s):
            raise SystemExit(f"ERRO SANITY {nome_var}: loss não finita")
        loss_s.backward()
        soma = 0.0
        for nome_p, p in cab.named_parameters():
            if p.grad is None or not torch.isfinite(p.grad).all():
                raise SystemExit(f"ERRO SANITY {nome_var}: gradiente ausente/inválido em {nome_p}")
            soma += float(p.grad.abs().sum().item())
        if soma <= 0:
            raise SystemExit(f"ERRO SANITY {nome_var}: gradientes todos zero")
        for nome_p, p in modelo.named_parameters():
            if p.requires_grad or p.grad is not None:
                raise SystemExit(f"ERRO SANITY: base recebeu gradiente em {nome_p}")
        torch.nn.utils.clip_grad_norm_(cab.parameters(), 1.0)
        opt.step()
    g, fim, _ = gerar_pointer(cab, DEV_EX[0]["prompt_ids"], max_novos=8)
    print(f"  {nome_var}: loss={loss_s.item():.3f} kl={kl_s.item():.4f} stop={met_s['loss_stop']:.3f} | smoke: {tokenizer.decode(g)!r}", flush=True)
    del cab, opt

if fingerprint_modulo(modelo) != HASH_ANTES:
    raise SystemExit("ERRO SANITY: hash da base mudou")
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("SANITY CHECK PASSOU — base intacta, gradiente só nas cabeças, geração não quebrou.", flush=True)
print("PRECHECK 7/9 OK", flush=True)


In [ ]:
# Célula 8 — teto de cada arquitetura + BASELINE congelado, antes de qualquer treino
print("PRECHECK 8/9 — teto e baseline", flush=True)

TETO = {
    "v3_mesma_cabeca": teto_teacher_forcing(BLIND_NOVO_EX),
    "AB_troca_parada": teto_teacher_forcing(BLIND_NOVO_EX, variante=True, parada=True),
    "ABC_troca_parada_adaptador": "sem teto estrutural: o adaptador pode mudar a ordem do vocabulário",
}
print("\n=== TETO sob teacher forcing (blind novo) ===", flush=True)
for nome_t, t in TETO.items():
    if isinstance(t, str):
        print(f"  {nome_t}: {t}", flush=True)
        continue
    print(f"  {nome_t}: cópia+fim ≤ {100*t['teto_exact_name_fim_copia']:.0f}% | nome alcançável: "
          + ", ".join(f"{k} {100*t[k]['nome_alcancavel']:.0f}%" for k in d3.TAREFAS), flush=True)

print("\n=== BASELINE — base congelada sem cabeça ===", flush=True)
BASE_NOVO_LINHAS = avaliar_conjunto("base", BLIND_NOVO_EX, rotulo="base/blind_novo")
BASE_NOVO = d3.resumir(BASE_NOVO_LINHAS)
BASE_V2REAL = d3.resumir(avaliar_conjunto("base", BLIND_V2_REAL_EX, rotulo="base/blind_v2_real"))
BASE_LEGADO = resumir_legado(avaliar_legado("base", BLIND_V2_LEGADO_EX, rotulo="base/blind_v2_legado"))
print("base blind novo: nome exato", f"{100*BASE_NOVO['exact_name']:.1f}%",
      "| semântica geral", f"{100*BASE_NOVO['semantica_geral']:.1f}%",
      "(v3 mediu 25,7% / 19,0%)", flush=True)
print("base blind v2 legado: nome", f"{100*BASE_LEGADO['exact_entity_match']:.1f}%",
      "(v2 e v3 mediram 3,3%)", flush=True)
print("PRECHECK 8/9 OK", flush=True)


In [ ]:
# Célula 9 — treino das 4 configurações (AB/ABC × peso da âncora), mesmos lotes e hiperparâmetros
PASSOS = 1000
BATCH = 16
N_NO_COPY = 4
LR = 2e-3
EVAL_CADA = 100
PACIENCIA = 4
DEV_MONITOR = DEV_EX[:72]  # 12 entidades de dev × 6 tarefas
N_ANCORA = 8
PRESERVACAO_12 = [
    "Explique em uma frase o que é gravidade.", "Qual é a função de uma cadeira?",
    "O que significa memória?", "Diga uma característica da água.", "O que é uma estrela?",
    "Qual é a função de um recipiente?", "Explique o que é temperatura.",
    "O que significa aprender?", "Diga uma função de uma ferramenta.", "O que é uma hipótese?",
    "Explique o que é energia.", "Qual é a diferença entre fato e opinião?",
]

@torch.no_grad()
def preservacao_next_token(pointer_obj, prompts=None):
    """Primeiro token de respostas genéricas, base vs cabeça (12 fixas do v2/v3 por padrão)."""
    prompts = PRESERVACAO_12 if prompts is None else prompts
    acordos, tvs = [], []
    for msg in prompts:
        prompt = lp.fonte_dialogo(tokenizer, msg, (), cfg.contexto)
        x = torch.tensor([prompt], dtype=torch.long, device=dispositivo)
        pb = probs_base(x)[0, -1]
        pl = torch.tensor([len(prompt)], dtype=torch.long, device=dispositivo)
        pp = distribuicao_final(pointer_obj, x, pl)[0][0, -1]
        acordos.append(int(torch.argmax(pb)) == int(torch.argmax(pp)))
        tvs.append(0.5 * torch.abs(pb - pp).sum().item())
    return {"top1_agreement": sum(acordos) / len(acordos),
            "mean_total_variation": sum(tvs) / len(tvs), "n": len(prompts)}

@torch.no_grad()
def dev_monitor(pointer_obj):
    pointer_obj.eval()
    linhas = []
    for ex in DEV_MONITOR:
        g, fim, _ = gerar_pointer(pointer_obj, ex["prompt_ids"], max_novos=max_novos_ex(ex))
        linhas.append({"tarefa": ex["tarefa"], **d3.avaliar_saida(ex, tokenizer.decode(g), fim)})
    por = {t: sum(r["semantic_ok"] for r in linhas if r["tarefa"] == t) / sum(r["tarefa"] == t for r in linhas)
           for t in d3.TAREFAS}
    pres = preservacao_next_token(pointer_obj, ANCORA_DEV_PROMPTS)  # escolha: âncoras de dev
    pres12 = preservacao_next_token(pointer_obj)  # só informativo; checagem final usa estas
    sem = sum(por.values()) / len(por)
    pointer_obj.train()
    return {**{f"sem_{t}": v for t, v in por.items()}, "sem_media": sem,
            "preservation_top1": pres["top1_agreement"], "preservation_tv": pres["mean_total_variation"],
            "preservation_ok": pres["top1_agreement"] >= 0.90 and pres["mean_total_variation"] <= 0.15,
            "pres12_top1": pres12["top1_agreement"], "pres12_tv": pres12["mean_total_variation"],
            "score": sem + 0.25 * pres["top1_agreement"]}

def treinar_variante(nome_var, indice):
    torch.manual_seed(SEED + 100 * (indice + 1))
    cab = nova_cabeca(nome_var)
    lambda_kl = CONFIGS[nome_var][1]
    rr_anc = random.Random(SEED + 8008)
    otim = torch.optim.AdamW(cab.parameters(), lr=LR, weight_decay=0.01)
    rr_train = random.Random(SEED + 4004)  # mesmos lotes para todas as variantes
    hist, melhor, sem_melhora, feitos, t0 = [], None, 0, 0, time.time()
    print(f"\n### {nome_var}: {n_params(cab):,} parâmetros treináveis", flush=True)
    cab.train()
    for passo in range(1, PASSOS + 1):
        batch = rr_train.sample(TREINO, BATCH - N_NO_COPY) + [rr_train.choice(NO_COPY_EX) for _ in range(N_NO_COPY)]
        rr_train.shuffle(batch)
        otim.zero_grad(set_to_none=True)
        loss, met = perda_batch(cab, batch, dispositivo)
        kl = perda_ancora(cab, rr_anc.sample(ANCORA_TREINO, N_ANCORA), dispositivo)
        met["kl_ancora"] = float(kl.item())
        loss = loss + lambda_kl * kl
        if not torch.isfinite(loss):
            raise SystemExit(f"ERRO: loss NaN/Inf em {nome_var} passo {passo}")
        loss.backward()
        for nome_p, p in cab.named_parameters():
            if p.grad is None or not torch.isfinite(p.grad).all():
                raise SystemExit(f"ERRO: gradiente inválido em {nome_var}.{nome_p}")
        torch.nn.utils.clip_grad_norm_(cab.parameters(), 1.0)
        otim.step()
        feitos = passo
        if passo == 1 or passo % 50 == 0:
            print(f"  passo {passo:4d} loss={loss.item():.3f} mix={met['loss_mix']:.3f} copy={met['loss_copy']:.3f} "
                  f"fim={met['loss_fim']:.3f} stop={met['loss_stop']:.3f} kl={met['kl_ancora']:.4f}", flush=True)
        if passo == 1 or passo % EVAL_CADA == 0:
            dm = dev_monitor(cab)
            hist.append({"passo": passo, "loss": float(loss.item()), **met, **dm, "segundos": round(time.time() - t0, 1)})
            print(f"  DEV passo={passo} " + " ".join(f"{t}={100*dm['sem_'+t]:.0f}%" for t in d3.TAREFAS)
                  + f" | âncoras top1={dm['preservation_top1']:.2f} TV={dm['preservation_tv']:.3f} | 12 fixas top1={dm['pres12_top1']:.2f} | score={dm['score']:.3f}", flush=True)
            elegivel = dm["preservation_ok"]
            if elegivel and (melhor is None or dm["score"] > melhor["score"] + 1e-4):
                melhor = {"score": dm["score"], "passo": passo, "dev": dm,
                          "state": {k: v.detach().cpu().clone() for k, v in cab.state_dict().items()}}
                sem_melhora = 0
            else:
                sem_melhora += 1
            if passo >= 300 and sem_melhora >= PACIENCIA:
                print("  EARLY STOP", flush=True)
                break
        if passo % 200 == 0 and fingerprint_modulo(modelo) != HASH_ANTES:
            raise SystemExit("ERRO: hash da base mudou durante o treino")
    if melhor is None:
        print(f"  {nome_var}: nenhum passo manteve a preservação; variante descartada", flush=True)
        return {"nome": nome_var, "cabeca": None, "melhor": None, "historico": hist, "passos": feitos}
    cab.load_state_dict(melhor["state"])
    cab.eval()
    print(f"  {nome_var}: melhor passo {melhor['passo']} score_dev={melhor['score']:.3f}", flush=True)
    return {"nome": nome_var, "cabeca": cab, "melhor": melhor, "historico": hist, "passos": feitos}

TREINADAS = {nome: treinar_variante(nome, i) for i, nome in enumerate(CONFIGS)}

if fingerprint_modulo(modelo) != HASH_ANTES:
    raise SystemExit("ERRO: Transformer mudou durante o treino")

# Escolha do candidato SÓ pelo dev, registrada antes de abrir o blind.
validas = [v for v in TREINADAS.values() if v["melhor"] is not None]
if not validas:
    raise SystemExit("ERRO: nenhuma variante preservou a base no dev")
CANDIDATO = max(validas, key=lambda v: v["melhor"]["score"])["nome"]
ESCOLHA_DEV = {v["nome"]: (None if v["melhor"] is None else
                           {"score": v["melhor"]["score"], "passo": v["melhor"]["passo"], "dev": v["melhor"]["dev"]})
               for v in TREINADAS.values()}
print("\nEscolha pelo dev (antes do blind):", CANDIDATO, flush=True)
print(json.dumps(ESCOLHA_DEV, ensure_ascii=False, indent=1), flush=True)


In [ ]:
# Célula 10 — avaliação final: todas as variantes no blind novo; candidato também nas regressões
print("\n=== AVALIAÇÃO FINAL v5 (só o candidato abre o blind) ===", flush=True)
FINAL = {}
_cand = TREINADAS[CANDIDATO]["cabeca"]
_linhas = avaliar_conjunto("pointer", BLIND_NOVO_EX, pointer_obj=_cand, rotulo=f"{CANDIDATO}/blind_novo")
FINAL[CANDIDATO] = {"linhas": _linhas, "resumo": d3.resumir(_linhas),
                    "preservacao": preservacao_next_token(_cand),
                    "preservacao_ancoras_dev": preservacao_next_token(_cand, ANCORA_DEV_PROMPTS)}

CAB = TREINADAS[CANDIDATO]["cabeca"]
CAND_V2REAL = d3.resumir(avaliar_conjunto("pointer", BLIND_V2_REAL_EX, pointer_obj=CAB, rotulo="candidato/blind_v2_real"))
CAND_LEGADO = resumir_legado(avaliar_legado("pointer", BLIND_V2_LEGADO_EX, pointer_obj=CAB, rotulo="candidato/blind_v2_legado"))

def pct(v):
    return "  —  " if v is None else f"{100*v:5.1f}%"

def tabela(base, final, rotulo):
    print(f"\n--- {rotulo}: base → cabeça ---", flush=True)
    for t in d3.TAREFAS:
        b, p = base[t], final[t]
        print(f"{t:12s} exact-name {pct(b['exact_name'])} → {pct(p['exact_name'])} | "
              f"exact-name+fim {pct(b['exact_name_fim'])} → {pct(p['exact_name_fim'])} | "
              f"semântica {pct(b['semantica'])} → {pct(p['semantica'])} | duplicado {pct(p['nome_duplicado'])}", flush=True)
    print(f"nome exato geral {pct(base['exact_name'])} → {pct(final['exact_name'])} | "
          f"close {pct(final['lexical_close'])} distant {pct(final['lexical_distant'])} | "
          f"semântica geral {pct(base['semantica_geral'])} → {pct(final['semantica_geral'])}", flush=True)

for nome_var, r in FINAL.items():
    marca = "  ← CANDIDATO (escolhido pelo dev)" if nome_var == CANDIDATO else ""
    tabela(BASE_NOVO, r["resumo"], f"{nome_var}{marca}")
    print("  preservação (12 fixas):", json.dumps(r["preservacao"], ensure_ascii=False), flush=True)
    print("  preservação (âncoras de dev):", json.dumps(r["preservacao_ancoras_dev"], ensure_ascii=False), flush=True)
tabela(BASE_V2REAL, CAND_V2REAL, f"blind v2 formato real — {CANDIDATO}")
print("\nblind v2 legado — base:", json.dumps(BASE_LEGADO, ensure_ascii=False), flush=True)
print(f"blind v2 legado — {CANDIDATO}:", json.dumps(CAND_LEGADO, ensure_ascii=False), flush=True)

if fingerprint_modulo(modelo) != HASH_ANTES:
    raise SystemExit("ERRO: hash da base mudou durante avaliação.")


In [ ]:
# Célula 11 — gates congelados no CANDIDATO (blind novo) + salvamento em pasta NOVA
print("PÓS-CHECK — gates e salvamento", flush=True)

HASH_DEPOIS = fingerprint_modulo(modelo)
BASE_UNCHANGED = HASH_DEPOIS == HASH_ANTES
if not BASE_UNCHANGED:
    raise SystemExit("ERRO: base mudou. Nada será salvo como resultado válido.")

FN = FINAL[CANDIDATO]["resumo"]
PRES = FINAL[CANDIDATO]["preservacao"]
pos_vals = [FN[f"posicao_{p}"] for p in d3.POSICOES]
GATES = {
    "exact_name_fim_copia>=0.70": FN["exact_name_fim_copia"] >= 0.70,
    "exact_name_distant>=0.65": FN["lexical_distant"] >= 0.65,
    "abs(close-distant)<=0.10": abs(FN["lexical_close"] - FN["lexical_distant"]) <= 0.10,
    "max(posicao)-min(posicao)<=0.10": max(pos_vals) - min(pos_vals) <= 0.10,
    "regressao_funcao<=5pp": BASE_NOVO["funcao"]["semantica"] - FN["funcao"]["semantica"] <= 0.05 + 1e-9,
    "regressao_semantica_geral<=2pp": BASE_NOVO["semantica_geral"] - FN["semantica_geral"] <= 0.02 + 1e-9,
    "transformer_hash_igual": BASE_UNCHANGED,
}
GATE_PASS = all(GATES.values())
EXTRA_CHECKS = {
    "preservation_top1>=0.90": PRES["top1_agreement"] >= 0.90,
    "preservation_mean_TV<=0.15": PRES["mean_total_variation"] <= 0.15,
}

def por_tarefa(r):
    return {t: {k: r[t][k] for k in ("exact_name", "exact_name_fim", "semantica", "resposta_exata", "nome_duplicado")}
            for t in d3.TAREFAS}

resultado = {
    "protocol": "pointer_copy_causal_v5",
    "created_at": datetime.now().isoformat(),
    "repo_branch": REPO_BRANCH, "repo_commit": commit,
    "dados_congelados_sha256": ESPERADO_SHA,
    "base_checkpoint": str(BASE_RUN), "base_phase": "05_05_dialogo_estado",
    "base_file_sha256": SHA_PESOS, "tokenizer_sha256": SHA_TOKENIZER,
    "base_hash_before": HASH_ANTES, "base_hash_after": HASH_DEPOIS, "base_unchanged": BASE_UNCHANGED,
    "base_parameters": int(N_BASE),
    "configuracoes": {n: {"arquitetura": CONFIGS[n][0], "config": VARIANTES[CONFIGS[n][0]], "lambda_kl": CONFIGS[n][1],
                          "parametros": n_params(nova_cabeca(n)), "passos": TREINADAS[n]["passos"],
                          "historico": TREINADAS[n]["historico"]} for n in CONFIGS},
    "ancoras_sha256": ANCORAS_SHA, "ancoras_treino": len(ANCORA_TREINO), "ancoras_dev": len(ANCORA_DEV_PROMPTS),
    "mapa_variante_trocas": int(N_TROCAS),
    "copiabilidade_v4_blind_novo": COPIABILIDADE,
    "teto_teacher_forcing_blind_novo": TETO,
    "escolha_pelo_dev": ESCOLHA_DEV,
    "candidato": CANDIDATO,
    "blind_novo_baseline": BASE_NOVO,
    "blind_novo_candidato": {"resumo": FN, "por_tarefa": por_tarefa(FN), "preservacao": PRES,
                             "preservacao_ancoras_dev": FINAL[CANDIDATO]["preservacao_ancoras_dev"]},
    "blind_v2_formato_real": {"baseline": BASE_V2REAL, "candidato": CAND_V2REAL},
    "blind_v2_legado": {"baseline": BASE_LEGADO, "candidato": CAND_LEGADO},
    "gates": GATES, "extra_checks": EXTRA_CHECKS, "gate_pass": bool(GATE_PASS),
    "porta_conversacional": "pendente — exigida mesmo se gate_pass=true",
    "ativo_no_chat": False, "promovido": False,
    "linhas_blind_novo_baseline": BASE_NOVO_LINHAS,
    "linhas_blind_novo_candidato": FINAL[CANDIDATO]["linhas"],
}

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DEST_ROOT = DRIVE_CHECKPOINTS / "experimento_pointer_copy_v5"
DEST_ROOT.mkdir(parents=True, exist_ok=True)
DEST = DEST_ROOT / f"pointer_copy_v5_{stamp}"
for antigo in ("experimento_pointer_copy/", "experimento_pointer_copy_v3/", "experimento_pointer_copy_v4/"):
    if antigo in str(DEST) + "/":
        raise SystemExit("ERRO: destino apontaria para a pasta de uma rodada anterior")
DEST.mkdir(parents=False, exist_ok=False)

for nome_var, v in TREINADAS.items():
    if v["cabeca"] is None:
        continue
    torch.save({
        "state_dict": {k: t.detach().cpu() for k, t in v["cabeca"].state_dict().items()},
        "protocol": "pointer_copy_causal_v5", "configuracao": nome_var, "arquitetura": CONFIGS[nome_var][0],
        "config": VARIANTES[CONFIGS[nome_var][0]], "lambda_kl": CONFIGS[nome_var][1],
        "rank": RANK, "dimensao": cfg.dimensao, "vocabulario": cfg.vocabulario,
        "mapa_variante": MAPA_VARIANTE_CPU,
        "base_checkpoint": str(BASE_RUN), "base_hash": HASH_DEPOIS,
        "base_file_sha256": SHA_PESOS, "tokenizer_sha256": SHA_TOKENIZER,
        "candidato": nome_var == CANDIDATO,
        "gate_pass": bool(GATE_PASS) if nome_var == CANDIDATO else None,
        "ativo_no_chat": False, "promovido": False,
    }, DEST / f"cabeca_v5_{nome_var}.pt")
(DEST / "resultado_pointer_copy_v5.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2), encoding="utf-8")

linhas_tab = "\n".join(
    f"{t:12s} exact-name {pct(BASE_NOVO[t]['exact_name'])} -> {pct(FN[t]['exact_name'])} | "
    f"exact-name+fim {pct(BASE_NOVO[t]['exact_name_fim'])} -> {pct(FN[t]['exact_name_fim'])} | "
    f"semantica {pct(BASE_NOVO[t]['semantica'])} -> {pct(FN[t]['semantica'])}"
    for t in d3.TAREFAS)
outras = "\n".join(
    f"  {n}: " + ("descartada (nenhum passo preservou as âncoras)" if v is None else
                  f"score {v['score']:.3f} no passo {v['passo']} | âncoras top1 {v['dev']['preservation_top1']:.2f}")
    for n, v in ESCOLHA_DEV.items())
readme = f"""CRIVO — pointer_copy_causal_v5
base: {BASE_RUN}
repo: {REPO_BRANCH} @ {commit}
base_file_sha256: {SHA_PESOS}
base_hash_before/after: {HASH_ANTES} / {HASH_DEPOIS}

configurações no dev (candidato escolhido pelo dev: {CANDIDATO}):
{outras}

candidato {CANDIDATO}, base -> cabeça:
{linhas_tab}
semantica_geral {pct(BASE_NOVO['semantica_geral'])} -> {pct(FN['semantica_geral'])}

gates:
{json.dumps(GATES, ensure_ascii=False, indent=2)}
extra_checks:
{json.dumps(EXTRA_CHECKS, ensure_ascii=False, indent=2)}

gate_pass: {GATE_PASS}
porta_conversacional: pendente
ativo_no_chat: false
promovido: false
"""
(DEST / "README_resultado_v5.txt").write_text(readme, encoding="utf-8")
for p in [DEST / "resultado_pointer_copy_v5.json", DEST / "README_resultado_v5.txt"] + list(DEST.glob("cabeca_v5_*.pt")):
    if not p.is_file() or p.stat().st_size == 0:
        raise SystemExit(f"ERRO de persistência: {p}")

print("\n================ RESULTADO v5 ================", flush=True)
print(readme, flush=True)
print("SALVO EM:", DEST, flush=True)
print(f"gate_pass={GATE_PASS} | porta conversacional pendente | ativo_no_chat=false | promovido=false", flush=True)
